# AIBackends - built-in tasks on the Hugging Face Transformers runtime

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/donvito/notebooks/blob/main/colab/AIBackends-Transformers-runtime-tasks.ipynb)

Run the same aibackends tasks you would run on llama.cpp, but through the
`TRANSFORMERS` runtime (`aibackends[transformers]`). Mirrors
`examples/tasks/basic_task_transformers.py`.

Any Hugging Face causal LM works: pass `ModelRef(name="<hf repo id>")`. This notebook uses
the ungated `Qwen/Qwen2.5-1.5B-Instruct` so it runs anywhere without a token. The
registered `GEMMA3_270M_IT` profile works too, but `google/gemma-3-270m-it` is gated, so
accept its license and add an `HF_TOKEN` Colab secret first.

**Runtime:** works on a CPU runtime; for faster inference pick *Runtime > Change runtime type > T4 GPU*. The device is detected automatically.

In [ ]:
%pip install -q "aibackends[transformers]>=0.8.1"

# If a later import fails with a transformers error, use
# Runtime > Restart session, then continue from the next cell.

In [2]:
import aibackends

# aibackends accepts: cpu, gpu, cuda, cuda:<index>, mps
try:
    import torch

    DEVICE = "gpu" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"

print("aibackends", aibackends.__version__)
print("device:", DEVICE)

aibackends 0.8.1
device: cpu


In [3]:
from pathlib import Path
from urllib.request import urlretrieve

DATA_URL = "https://raw.githubusercontent.com/donvito/aibackends/main/examples/data"
DATA_DIR = Path("aibackends_data")


def fetch(relative_path: str) -> Path:
    """Download a sample file from the aibackends examples once and return its path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve(f"{DATA_URL}/{relative_path}", path)
    return path

## 1. Structured invoice extraction

`device` maps to the Transformers `device_map` (`gpu` -> `cuda`). On GPU we also load the
weights in float16 to halve memory.

In [4]:
from aibackends.models import ModelRef
from aibackends.runtimes import TRANSFORMERS
from aibackends.tasks import ExtractInvoiceTask, create_task

MODEL = ModelRef(name="Qwen/Qwen2.5-1.5B-Instruct")
RUNTIME_OPTIONS = {
    "runtime": TRANSFORMERS,
    "model": MODEL,
    "device": DEVICE,
    "extra_options": {"dtype": "float16"} if DEVICE == "gpu" else {},
}

invoice_task = create_task(ExtractInvoiceTask, max_tokens=512, **RUNTIME_OPTIONS)
invoice = invoice_task.run(fetch("invoice.txt"))
print(invoice.model_dump_json(indent=2))

{
  "vendor": "Acme Corp",
  "line_items": [
    {
      "description": "AI integration consulting",
      "quantity": 1.0,
      "unit_price": 1250.0,
      "amount": 1250.0
    }
  ],
  "subtotal": 1250.0,
  "tax": 0.0,
  "total": 1250.0,
  "due_date": null,
  "payment_terms": "Net 30"
}


## 2. Summarize and classify with the same runtime

The runtime is reused across tasks with identical config, so the model loads once.

In [5]:
from aibackends.tasks import classify, summarize

notes = fetch("meeting_notes.txt")
print(summarize(notes, max_tokens=256, **RUNTIME_OPTIONS))

label = classify(
    fetch("contract.txt"),
    labels=["invoice", "rental contract", "employment contract", "receipt", "sales call"],
    prompt='Respond with JSON keys "label", "confidence", and "all_scores".',
    max_tokens=256,
    **RUNTIME_OPTIONS,
)
print(label.model_dump_json(indent=2))

The AIBacksend team is preparing for their upcoming launch by finalizing the README file to emphasize a "local-first" approach. They plan to record a brief demonstration of extracting invoices using the platform. The team will also publish a detailed build log on both X and LinkedIn to showcase their development progress. Additionally, they have reached out to two design firms specializing in fintech and logistics to collaborate on visual elements for the product. Finally, they are developing a comprehensive talk outline for a meetup event scheduled in Singapore focused on artificial intelligence.


{
  "label": "rental contract",
  "confidence": 1.0,
  "all_scores": {
    "invoice": 0.0,
    "rental contract": 1.0,
    "employment contract": 0.0,
    "receipt": 0.0,
    "sales call": 0.0
  }
}


## 3. Raw chat completions

`get_runtime` exposes the underlying runtime for free-form prompts.

In [6]:
from aibackends import get_runtime

runtime = get_runtime({**RUNTIME_OPTIONS, "max_tokens": 128})
reply = runtime.complete([
    {"role": "system", "content": "Answer in one sentence."},
    {"role": "user", "content": "Why run small language models locally?"},
])
print(reply.content)

Running small language models locally allows for better control over data and privacy, as well as the ability to fine-tune them without relying on cloud services.
